# Experiment 1: Data Collection

| Field | Value |
|---|---|
| **Name** | Francis Reuben R |
| **USN** | 1RUA25CSE0168 |
| **Section** | G |
| **Course** | Data Science Laboratory |
| **Semester** | 3 |
| **Experiment** | 1 |

## Web scraping: bookstore catalogue and quotes


### Objective

Scrape structured records from HTML pages that have no JSON API.

1. **books.toscrape.com** (required): title, price, star rating across every catalogue page. Save `books_data.csv`.
2. **quotes.toscrape.com** (practice from the lab session): quote text and author. Save `quotes.csv`.

### Rules before scraping

1. Check `robots.txt`.
2. Check the site terms. Indian property portals (99acres, MagicBricks, NoBroker) do not allow automated collection.
3. Wait between requests. Do not hammer the server.
4. Do not collect personal data (names, phones, emails).

This lab uses practice sites that exist for scraping tutorials.


### Setup


In [1]:
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.robotparser import RobotFileParser

STUDENT = {
    "name": "Francis Reuben R",
    "usn": "1RUA25CSE0168",
    "section": "G",
}
HEADERS = {
    "User-Agent": "DS-Lab-Exp1/1.0 (Francis Reuben R; USN 1RUA25CSE0168; Section G; academic)"
}

def can_fetch(url: str) -> bool:
    parts = url.split("/")
    robots_url = f"{parts[0]}//{parts[2]}/robots.txt"
    parser = RobotFileParser()
    parser.set_url(robots_url)
    try:
        parser.read()
        return parser.can_fetch("*", url)
    except Exception as exc:
        print("robots.txt check failed:", exc)
        return False

print(STUDENT)
print("books.toscrape robots allow:", can_fetch("https://books.toscrape.com/catalogue/page-1.html"))
print("quotes.toscrape robots allow:", can_fetch("https://quotes.toscrape.com/"))


{'name': 'Francis Reuben R', 'usn': '1RUA25CSE0168', 'section': 'G'}


books.toscrape robots allow: True


quotes.toscrape robots allow: True


### Bookstore scrape

Follow the `Next` link until the last page. Each book sits in `article.product_pod`. Price is parsed as a float by keeping digits and the decimal point (avoids the `Â£` encoding trap).


In [2]:
def scrape_books():
    base_url = "https://books.toscrape.com/catalogue/"
    next_page = "page-1.html"
    rows = []
    page_count = 0

    while next_page:
        page_count += 1
        url = requests.compat.urljoin(base_url, next_page)
        print(f"Scraping {url}")
        response = None
        for attempt in range(1, 6):
            response = requests.get(url, headers=HEADERS, timeout=30)
            if response.status_code in (429, 500, 502, 503, 504):
                time.sleep(1.5 * attempt)
                continue
            break
        response.raise_for_status()
        soup = BeautifulSoup(response.content, "html.parser")

        for book in soup.find_all("article", class_="product_pod"):
            title = book.h3.a["title"]
            price_text = book.find("p", class_="price_color").get_text(strip=True)
            price = float("".join(ch for ch in price_text if ch.isdigit() or ch == "."))
            rating = book.find("p", class_="star-rating")["class"][1]
            rows.append({"title": title, "price": price, "rating": rating})

        next_button = soup.find("li", class_="next")
        next_page = next_button.a["href"] if next_button else None
        time.sleep(0.25)

    print(f"Scraped {len(rows)} books from {page_count} pages.")
    return pd.DataFrame(rows)


books_df = scrape_books()
books_df.head()


Scraping https://books.toscrape.com/catalogue/page-1.html


Scraping https://books.toscrape.com/catalogue/page-2.html


Scraping https://books.toscrape.com/catalogue/page-3.html


Scraping https://books.toscrape.com/catalogue/page-4.html


Scraping https://books.toscrape.com/catalogue/page-5.html


...

Scraping https://books.toscrape.com/catalogue/page-48.html


Scraping https://books.toscrape.com/catalogue/page-49.html


Scraping https://books.toscrape.com/catalogue/page-50.html


Scraped 1000 books from 50 pages.


,title,price,rating
0,A Light in the Attic,51.77,Three
1,Tipping the Velvet,53.74,One
2,Soumission,50.10,One
3,Sharp Objects,47.82,Four
4,Sapiens: A Brief History of Humankind,54.23,Five


In [3]:
books_path = "books_data.csv"
books_df.to_csv(books_path, index=False, encoding="utf-8")
print("Saved", books_path, "rows:", len(books_df))
print(books_df["rating"].value_counts())
print("Price range:", books_df["price"].min(), "to", books_df["price"].max())


Saved books_data.csv rows: 1000
rating
One      226
Three    203
Five     196
Two      196
Four     179
Name: count, dtype: int64
Price range: 10.0 to 59.99


### Practice: quotes.toscrape.com

Each quote is a `div.quote`. Text is in `span.text`, author in `small.author`.


In [4]:
quotes_url = "https://quotes.toscrape.com/"
response = requests.get(quotes_url, headers=HEADERS, timeout=30)
if response.status_code in (429, 500, 502, 503, 504):
    time.sleep(2)
    response = requests.get(quotes_url, headers=HEADERS, timeout=30)
response.raise_for_status()
soup = BeautifulSoup(response.text, "html.parser")

quote_rows = []
for quote in soup.find_all("div", class_="quote"):
    quote_rows.append({
        "quote": quote.find("span", class_="text").get_text(strip=True),
        "author": quote.find("small", class_="author").get_text(strip=True),
    })

quotes_df = pd.DataFrame(quote_rows)
quotes_df.to_csv("quotes.csv", index=False, encoding="utf-8")
print("Saved quotes.csv rows:", len(quotes_df))
quotes_df.head()


Saved quotes.csv rows: 10


,quote,author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


### Note on real-estate sites

A class mini-project asked for listings from 99acres / NoBroker / MagicBricks (location, sqft, BHK, bathrooms, balconies, area type, price). Those sites forbid automated collection. Do not scrape them live. If a faculty-provided local HTML snapshot is available, parse that file offline and drop personal fields (agent name, phone).

### Result

`books_data.csv` holds the full practice catalogue. `quotes.csv` holds the first quotes page.

### Conclusion

Paginated HTML was parsed with BeautifulSoup. Title, price, and rating were extracted from `books.toscrape.com` across all catalogue pages and saved as CSV. A short delay was used between page requests.
